# DVC from zero — the hands-on half

This notebook is the **practical companion to `dvc_slides.html`**. The deck explains *why*;
this notebook makes it happen on your machine. Run the cells **top to bottom**.

| Notebook part | Deck slides | What you do |
|---|---|---|
| Part 0 · Setup | 16–17 | install DVC, build a throwaway repo |
| Part 1 · Version data | 19–26 | `dvc add`, remotes, push/pull, time travel |
| Part 2 · Pipelines | 27–31 | `dvc.yaml`, `dvc repro`, metrics |
| Part 3 · DVCLive | 32–33 | log metrics from inside training code |
| Part 4 · Advanced | 38–50 | experiments, run cache, `.dvcignore`, `artifacts:` |
| Part 5 · The real dataset | 3–6 | track the actual 39 MB `data.zip` from the deck |

**Nothing outside this folder is touched.** Everything happens in a throwaway `dvc_demo/`
created in Part 0; there is a cleanup cell at the end.

Lines starting with `!` are shell commands — Jupyter runs them in a terminal for you.

> **The one idea to hold onto:** a *hash* is a 32-character fingerprint of a file's bytes.
> DVC puts the fingerprint in Git and keeps the bytes somewhere else. Everything below is
> bookkeeping around that sentence.

---
> **Why bother?** Deck slides 3–15 make the case. Short version: by Part 5 of this notebook
> you will have 1,800 photographs under version control in a repository that is 61 KB.

---
# Part 0 — Setup

## Step 0.1 · Install DVC

* `dvc` — the tool itself
* `scikit-learn`, `pandas`, `numpy`, `pyyaml` — so we have a real pipeline to version

DVC is a **command-line tool**, not a service: there is nothing to run in the background
and no account to create. For a real project you would also pick a storage backend:

```bash
pip install "dvc[s3]"      # AWS S3 / MinIO
pip install "dvc[gs]"      # Google Cloud Storage
pip install "dvc[azure]"   # Azure Blob
pip install "dvc[gdrive]"  # Google Drive
pip install "dvc[ssh]"     # any SSH box or NAS
pip install "dvc[all]"     # everything
```

We use a plain folder as the remote below, so the bare install is enough.

### The whole idea, in one picture

<svg width="100%" viewBox="0 0 820 210" xmlns="http://www.w3.org/2000/svg" style="max-width:820px">
  <defs><marker id="p1" markerWidth="9" markerHeight="9" refX="8" refY="3" orient="auto">
    <path d="M0,0 L0,6 L9,3 z" fill="#945dd6"/></marker></defs>
  <rect x="8" y="30" width="230" height="150" rx="10" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="26" y="58" font-family="sans-serif" font-size="15" font-weight="700" fill="#24292f">Git repository</text>
  <text x="26" y="86"  font-family="monospace" font-size="12" fill="#57606a">train.py          2 KB</text>
  <text x="26" y="108" font-family="monospace" font-size="12" fill="#57606a">dvc.yaml          1 KB</text>
  <text x="26" y="130" font-family="monospace" font-size="12" fill="#945dd6">raw.csv.dvc      88 B</text>
  <text x="26" y="160" font-family="sans-serif" font-size="12" fill="#1a7f37">tiny — clones instantly</text>

  <path d="M246 105 H358" stroke="#945dd6" stroke-width="2" marker-end="url(#p1)"/>
  <text x="252" y="97" font-family="sans-serif" font-size="12" fill="#945dd6">points to a hash</text>

  <rect x="366" y="30" width="220" height="150" rx="10" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="384" y="58" font-family="sans-serif" font-size="15" font-weight="700" fill="#24292f">.dvc/cache</text>
  <text x="384" y="86"  font-family="monospace" font-size="12" fill="#57606a">files/md5/aa/0ebaab...</text>
  <text x="384" y="108" font-family="sans-serif" font-size="12" fill="#57606a">the real bytes</text>
  <text x="384" y="136" font-family="sans-serif" font-size="12" fill="#57606a">git-ignored</text>

  <path d="M594 105 H706" stroke="#945dd6" stroke-width="2" marker-end="url(#p1)"/>
  <text x="600" y="97" font-family="monospace" font-size="12" fill="#945dd6">dvc push</text>

  <rect x="714" y="30" width="98" height="150" rx="10" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="732" y="58" font-family="sans-serif" font-size="14" font-weight="700" fill="#24292f">Remote</text>
  <text x="732" y="86"  font-family="monospace" font-size="12" fill="#57606a">S3 / GCS</text>
  <text x="732" y="108" font-family="monospace" font-size="12" fill="#57606a">SSH / NAS</text>
  <text x="732" y="130" font-family="monospace" font-size="12" fill="#57606a">a folder</text>
</svg>

**Git carries the map. DVC carries the treasure.**

In [ ]:
!pip install -q dvc scikit-learn pandas numpy pyyaml

In [ ]:
!dvc --version        # a version number here means you are ready
!dvc doctor           # environment report: OS, python, filesystem, remotes

## Step 0.2 · Create the sandbox

We make a fresh folder that we can delete later without regret:

```
dvc_demo/       <- our fake "project", a real git repo
dvc_remote/            <- pretend cloud storage (just a folder on disk)
```

`dvc_remote/` being a plain folder is the point: **a DVC remote is anywhere that can hold
files.** S3, Google Drive, an SSH box, or a directory. The commands are identical.

In [ ]:
import os, shutil, pathlib

BASE   = pathlib.Path.cwd()                 # the folder this notebook lives in ("1/")
PROJ   = BASE / "dvc_demo"           # our sandbox git+dvc repo
REMOTE = BASE / "dvc_remote"                # our fake remote storage

# Start clean every time this cell runs, so re-running the notebook is safe.
for p in (PROJ, REMOTE):
    if p.exists():
        shutil.rmtree(p)
PROJ.mkdir(parents=True)
(PROJ / "src").mkdir()                      # our pipeline scripts will go here
REMOTE.mkdir(parents=True)

os.chdir(PROJ)                              # everything from here on happens inside the sandbox
print("working inside:", os.getcwd())

## Step 0.3 · Make a dataset

Real projects have a CSV that arrived from somewhere. We generate one so the notebook
needs no downloads: 2,000 rows of fake customer data with a `churn` label to predict.

Note `np.random.default_rng(42)` — the **seed**. Same seed, same numbers, every time.
That is the same reproducibility instinct DVC enforces at the project level.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)             # seeded -> everyone gets identical data
n = 2000

df = pd.DataFrame({
    "tenure_months":  rng.integers(1, 72, n),
    "monthly_charge": rng.normal(65, 20, n).round(2),
    "support_calls":  rng.poisson(1.2, n),
    "is_premium":     rng.integers(0, 2, n),
})

# A learnable rule + noise, so the model has a real signal to find.
score = (-0.04 * df.tenure_months + 0.03 * df.monthly_charge
         + 0.55 * df.support_calls - 0.8 * df.is_premium + rng.normal(0, 0.8, n))
df["churn"] = (score > score.mean()).astype(int)

os.makedirs("data", exist_ok=True)
df.to_csv("data/raw.csv", index=False)      # this is the file we are about to version

print(df.shape, "rows x cols;  churn rate:", df.churn.mean().round(3))
df.head()

---
# Part 1 — Versioning data   ·   deck slides 19–26

## Step 1.1 · `git init` — because DVC lives inside Git

DVC is not a replacement for Git. It is a **companion**: Git versions your code and the small
pointer files, DVC versions the big files those pointers point at.

`git config user.email/name` is set **locally** (this sandbox only) because a fresh machine
often has no identity configured, and `git commit` refuses to run without one.

In [ ]:
!git init -q                                        # -q = quiet
!git config user.email "trainee@qafza.local"        # local to THIS repo only
!git config user.name  "Qafza Trainee"

# Python bytecode should never be committed.
open(".gitignore", "w").write("__pycache__/\n")

!git add .gitignore && git commit -q -m "chore: initial commit"
!git log --oneline

# Remember the branch name ("main" or "master" depending on your git version) --
# we come back to it after time-travelling later.
BRANCH = !git rev-parse --abbrev-ref HEAD
BRANCH = BRANCH[0]
print("branch:", BRANCH)

## Step 1.2 · `dvc init`

One command. Look at what it creates — this is the whole DVC installation inside a project.

In [ ]:
!dvc init -q
!dvc config core.analytics false     # optional: turn off DVC's usage telemetry

print("--- .dvc/ ---")
!ls -a .dvc
print("--- git sees ---")
!git status --short

### What each piece is

| Path | What it is | In Git? |
|---|---|---|
| `.dvc/config` | your remotes and settings | **yes** — teammates need it |
| `.dvc/cache/` | the actual bytes of every version of every tracked file | **no** — machine-local |
| `.dvc/.gitignore` | written by DVC so `cache/` stays out of Git | yes |
| `.dvcignore` | like `.gitignore`, but tells DVC what not to scan | yes |

Notice `git status` already shows the config files staged for you. Commit them.

In [ ]:
!git commit -q -m "chore: dvc init"
!git log --oneline

## Step 1.3 · `dvc add` — the command that matters

This is **the** DVC command. Watch what it does to the folder.

In [ ]:
!dvc add data/raw.csv

Three things just happened. Let's prove each one.

**(1) A tiny pointer file was created.** Open it — it is human-readable YAML:

In [ ]:
!cat data/raw.csv.dvc

`md5` is the fingerprint of the CSV's bytes. `size` is how big it is. `path` is where the
file belongs. That is ~120 bytes of text standing in for the entire dataset — and text is
exactly what Git is good at.

**(2) Git was told to ignore the real file**, so it can never bloat the repo:

In [ ]:
!cat data/.gitignore

**(3) The bytes were copied into the cache**, filed under the first two characters of the
hash (a classic trick to avoid one directory with a million files in it):

In [ ]:
!find .dvc/cache -type f | head

## Step 1.4 · The two-command habit

Whenever data changes:

```
dvc add <file>     # DVC records the new bytes and updates the pointer
git add <file>.dvc # Git records the new pointer
git commit
```

Say it out loud a few times. It is 90% of daily DVC.

In [ ]:
!git add data/raw.csv.dvc data/.gitignore
!git commit -q -m "data: add raw.csv v1"
!git log --oneline

# Sanity check: how big is the repo Git is actually carrying?
print("\ncsv on disk:", os.path.getsize("data/raw.csv"), "bytes")
print("what git stores instead:", os.path.getsize("data/raw.csv.dvc"), "bytes")

## Step 1.5 · A remote, and `dvc push`

A **remote** is where the cache gets shared from. We use a local folder here; in production
you would write `s3://bucket/path`, `gs://…`, `ssh://…`, or `gdrive://…` instead. The
commands after this point do not change at all.

`-d` makes it the *default* remote so plain `dvc push` knows where to go.

In [ ]:
!dvc remote add -d storage "{REMOTE}"      # {REMOTE} is substituted by Jupyter from Python
!cat .dvc/config

!git add .dvc/config
!git commit -q -m "chore: add default dvc remote"
!dvc push          # upload the cache -> remote.  Like `git push`, but for data.
print("--- what landed in the remote ---")
!find "{REMOTE}" -type f | head

## Step 1.6 · Prove it works: delete the data and get it back

This is the moment DVC clicks for most people. We delete the CSV **and** the local cache —
simulating a teammate who has just cloned the repo and has no data at all. Then `dvc pull`.

In [ ]:
import shutil
os.remove("data/raw.csv")          # delete the dataset
shutil.rmtree(".dvc/cache")        # and the local cache: nothing left on this machine

print("raw.csv exists?", os.path.exists("data/raw.csv"))
print("but the pointer survived, because it is in Git:")
!ls data/

In [ ]:
!dvc pull          # remote -> cache -> your folder

print("raw.csv exists?", os.path.exists("data/raw.csv"))
print(pd.read_csv("data/raw.csv").shape, "-- identical rows, byte for byte")

> **Onboarding a teammate is now two commands:** `git clone <repo>` then `dvc pull`.
> They get the code and the *exact* matching data. No Drive links, no "which file did you mean".

## Step 1.7 · Time travel

Now the real payoff. We will:

1. change the dataset (append 500 new rows — "March data arrived"),
2. commit it as **v2**,
3. jump back to **v1**,
4. and jump forward to v2 again.

Remember: `git checkout` moves the *pointers*, `dvc checkout` moves the *data*.
Forgetting the second command is the #1 beginner mistake.

In [ ]:
# --- create v2 of the dataset ---
extra = df.sample(500, random_state=1)                    # pretend these are new records
pd.concat([df, extra]).to_csv("data/raw.csv", index=False)

!dvc status          # DVC compares the file's hash to the pointer and notices the change
!dvc add data/raw.csv                       # re-hash, update the pointer
!git add data/raw.csv.dvc
!git commit -q -m "data: add March rows (v2)"
!dvc push                                   # share the new version too

!git log --oneline
print("\nrows now:", len(pd.read_csv("data/raw.csv")))
print("pointer now:")
!grep md5 data/raw.csv.dvc

In [ ]:
# --- travel back to v1 ---
V1 = !git rev-parse HEAD~1                  # capture the previous commit's id into a Python list
V1 = V1[0]
V2 = !git rev-parse HEAD
V2 = V2[0]
print("v1 =", V1[:8], " v2 =", V2[:8])

!git checkout -q {V1}                       # code + pointers go back
print("\nafter git checkout only ->", len(pd.read_csv("data/raw.csv")), "rows (STILL v2 data!)")

!dvc checkout                               # now the data follows the pointer
print("after dvc checkout      ->", len(pd.read_csv("data/raw.csv")), "rows (this is v1)")

See the trap in the output above? After `git checkout` alone the file on disk was still the
**new** data while the pointer said **old**. Only `dvc checkout` reconciles them.

`dvc checkout` is near-instant even for huge files because it *links* from the cache instead
of copying. Ten branches sharing a 50 GB dataset cost you 50 GB once, not 500.

Let's return to the latest version and carry on.

In [ ]:
!git checkout -q {BRANCH}      # back to the branch tip, not a detached commit
!dvc checkout -q
print("back on", BRANCH, "with", len(pd.read_csv("data/raw.csv")), "rows")

---
# Part 2 — Pipelines   ·   deck slides 27–31

Versioning the data is step one. Step two is to stop keeping the recipe in your head.

A **pipeline** is a `dvc.yaml` listing *stages*. Each declares `cmd` (what to run), `deps`
(inputs — if a hash changes the stage is stale), `outs` (outputs DVC tracks for you), `params`
and `metrics`. Then `dvc repro` runs **only the stale stages**.

### The shape of what we are about to build

<svg width="100%" viewBox="0 0 830 100" xmlns="http://www.w3.org/2000/svg" style="max-width:830px">
  <defs><marker id="d1" markerWidth="9" markerHeight="9" refX="8" refY="3" orient="auto">
    <path d="M0,0 L0,6 L9,3 z" fill="#945dd6"/></marker></defs>
  <rect x="6"   y="22" width="132" height="52" rx="8" fill="#f6f8fa" stroke="#d0d7de"/>
  <text x="24"  y="45" font-family="monospace" font-size="12" fill="#24292f">data/raw.csv</text>
  <text x="24"  y="64" font-family="sans-serif" font-size="11" fill="#57606a">dvc add</text>
  <path d="M143 48 H196" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <rect x="203" y="22" width="112" height="52" rx="8" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="230" y="45" font-family="monospace" font-size="13" fill="#945dd6">prepare</text>
  <text x="218" y="64" font-family="sans-serif" font-size="11" fill="#57606a">prepare.py</text>
  <path d="M320 48 H373" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <rect x="380" y="22" width="142" height="52" rx="8" fill="#f6f8fa" stroke="#d0d7de"/>
  <text x="396" y="45" font-family="monospace" font-size="12" fill="#24292f">data/prepared/</text>
  <text x="396" y="64" font-family="sans-serif" font-size="11" fill="#57606a">train.csv + test.csv</text>
  <path d="M527 48 H580" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <rect x="587" y="22" width="100" height="52" rx="8" fill="#f6f8fa" stroke="#945dd6"/>
  <text x="612" y="45" font-family="monospace" font-size="13" fill="#945dd6">train</text>
  <text x="600" y="64" font-family="sans-serif" font-size="11" fill="#57606a">+ params.yaml</text>
  <path d="M692 48 H735" stroke="#945dd6" stroke-width="2" marker-end="url(#d1)"/>

  <text x="742" y="43" font-family="monospace" font-size="12" fill="#24292f">model.pkl</text>
  <text x="742" y="62" font-family="monospace" font-size="12" fill="#24292f">metrics.json</text>
</svg>

`dvc repro` walks this graph and runs only the boxes whose inputs changed.

## Step 2.1 · The knobs file, `params.yaml`

Hyperparameters live in a file instead of being buried in the script. DVC watches it, so
changing a number here is enough to make `dvc repro` retrain.

In [ ]:
%%writefile params.yaml
test_size: 0.2
random_state: 42
n_estimators: 60
max_depth: 4

## Step 2.2 · The stage scripts

`%%writefile` dumps the cell's contents to a file instead of executing it. We write two
plain Python scripts — nothing DVC-specific inside them, which is the point: **DVC wraps
ordinary scripts, it does not invade them.**

In [ ]:
%%writefile src/prepare.py
"""Stage 1: read the raw CSV, split into train/test, save both."""
import pandas as pd, yaml, os
from sklearn.model_selection import train_test_split

p = yaml.safe_load(open("params.yaml"))          # read the knobs

df = pd.read_csv("data/raw.csv")
train, test = train_test_split(df, test_size=p["test_size"],
                               random_state=p["random_state"], stratify=df.churn)

os.makedirs("data/prepared", exist_ok=True)
train.to_csv("data/prepared/train.csv", index=False)
test.to_csv("data/prepared/test.csv", index=False)
print(f"prepare: {len(train)} train / {len(test)} test rows")

In [ ]:
%%writefile src/train.py
"""Stage 2: train a random forest, save the model and a metrics file."""
import pandas as pd, yaml, json, os, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

p = yaml.safe_load(open("params.yaml"))

train = pd.read_csv("data/prepared/train.csv")
test  = pd.read_csv("data/prepared/test.csv")
FEATURES = ["tenure_months", "monthly_charge", "support_calls", "is_premium"]

model = RandomForestClassifier(n_estimators=p["n_estimators"],
                               max_depth=p["max_depth"],
                               random_state=p["random_state"])
model.fit(train[FEATURES], train.churn)

pred = model.predict(test[FEATURES])
prob = model.predict_proba(test[FEATURES])[:, 1]
metrics = {"accuracy": round(accuracy_score(test.churn, pred), 4),
           "f1":       round(f1_score(test.churn, pred), 4),
           "roc_auc":  round(roc_auc_score(test.churn, prob), 4)}

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/model.pkl")
json.dump(metrics, open("metrics.json", "w"), indent=2)
print("train:", metrics)

## Step 2.3 · Wire them into `dvc.yaml`

Read the `deps`/`outs` carefully: `prepare` **produces** `data/prepared`, and `train`
**depends on** it. That shared path is what makes DVC infer the order — you never write the
order down yourself.

In [ ]:
%%writefile dvc.yaml
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
      - src/prepare.py
      - data/raw.csv          # our dvc-tracked dataset
    params:
      - test_size
      - random_state
    outs:
      - data/prepared         # DVC tracks this folder for us

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/prepared         # output of `prepare` -> input here. This builds the DAG.
    params:
      - n_estimators
      - max_depth
      - random_state
    outs:
      - models/model.pkl
    metrics:
      - metrics.json:
          cache: false        # small + JSON -> keep it in Git so we can diff it in PRs

In [ ]:
!dvc dag        # DVC works out the order from deps/outs and draws it

## Step 2.4 · `dvc repro` — run it

First run: everything is stale, so both stages execute.

In [ ]:
!dvc repro

Now run it **again without changing anything**. Nothing should execute — DVC compares
hashes of every dep and param and skips what is unchanged.

In [ ]:
!dvc repro

### `dvc.lock` — the receipt

`dvc repro` wrote a lock file recording the exact hash of every input and output. This is
the machine-readable answer to *"which data and which parameters produced this model?"*.
**Commit it.**

In [ ]:
!cat dvc.lock

## Step 2.5 · Change one parameter and watch what re-runs

Bump `n_estimators` from 60 to 150. `prepare` does not depend on it, so DVC must skip
`prepare` and re-run only `train`.

In [ ]:
cfg = open("params.yaml").read().replace("n_estimators: 60", "n_estimators: 150")
open("params.yaml", "w").write(cfg)

!dvc repro      # look closely: 'prepare' is skipped, 'train' runs
!dvc metrics show

## Step 2.6 · Comparing metrics across commits

`dvc metrics diff` is what turns "the model got better" into a number a reviewer can see.
Let's commit this state, then ask DVC how it compares to the previous commit.

In [ ]:
!git add dvc.yaml dvc.lock params.yaml metrics.json src .gitignore
!git commit -q -m "feat: add dvc pipeline (n_estimators=150)"

# tweak again so there is something to diff against.
# Read FIRST, then write -- open(..., "w") empties the file immediately.
cfg = open("params.yaml").read().replace("max_depth: 4", "max_depth: 8")
open("params.yaml", "w").write(cfg)
!dvc repro -q
!dvc metrics diff HEAD

In `dvc/cml-example.yaml` (shipped with this repo) this exact command runs inside a GitHub
Action and posts the table as a **pull request comment**, alongside `dvc plots diff` charts.
That is CML — model review becomes code review.

In [ ]:
!git add -A && git commit -q -m "exp: max_depth 8"
!git log --oneline

---
# Part 3 — DVCLive   ·   deck slides 32–33

## Step 3.1 · Logging from inside your training code

Everything so far has been the CLI. **DVCLive** is the Python half: a few lines in your
training loop and each run becomes a DVC experiment that `dvc exp show` and `dvc plots show`
already understand.

This is the piece people miss — it means you do **not** need a separate tracking service to
compare runs.

In [ ]:
# commit the pipeline state so DVCLive's experiment snapshot is clean
# commit only if there is something to commit (keeps the output clean either way)
!git add -A && (git diff --cached --quiet || git commit -q -m "chore: before dvclive")
!pip install -q dvclive

In [ ]:
from dvclive import Live
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

train = pd.read_csv("data/prepared/train.csv")
test  = pd.read_csv("data/prepared/test.csv")
FEATURES = ["tenure_months", "monthly_charge", "support_calls", "is_premium"]

# save_dvc_exp=True -> when the block exits, this run is saved as a DVC experiment
with Live(save_dvc_exp=True, exp_name="dvclive-demo") as live:
    live.log_param("max_depth", 6)                 # an input you chose
    live.log_params({"random_state": 42, "logger": "dvclive"})

    # pretend each iteration is an epoch: log a metric per step
    for n in [20, 40, 80, 160]:
        m = RandomForestClassifier(n_estimators=n, max_depth=6,
                                   random_state=42).fit(train[FEATURES], train.churn)
        pred = m.predict(test[FEATURES])
        live.log_metric("roc_auc",  roc_auc_score(test.churn,
                                                  m.predict_proba(test[FEATURES])[:, 1]))
        live.log_metric("accuracy", accuracy_score(test.churn, pred))
        live.next_step()                           # advance the step counter

    live.log_sklearn_plot("confusion_matrix", test.churn.tolist(), pred.tolist())

Look at what it wrote. Nothing here is a database — it is plain files in the repo.

In [ ]:
!find dvclive -type f | sort
!cat dvclive/metrics.json          # the latest value of every metric
print("--- one row per step, ready for dvc plots ---")
!head -3 dvclive/plots/metrics/roc_auc.tsv

print("--- DVCLive also registered its plots in dvc.yaml ---")
!grep -A4 "^plots:" dvc.yaml

And the run shows up in the same table as the `dvc exp run` experiments from Step 3.3 —
one place for everything, whether the numbers came from the CLI or from Python.

In [ ]:
!dvc exp show --only-changed

> **Reference:** deck slide 33 lists every `Live()` argument (`monitor_system`, `report`,
> `resume`, …) and the framework callbacks for Keras, Lightning, HuggingFace, XGBoost and more.

---
# Part 4 — Advanced   ·   deck slides 38–50

The 10% you will want in month two. Every step here runs in the sandbox you just built.

## Step 4.1 · Read any version of the data straight from Python

`dvc.api` reads a tracked file at **any git revision** without checking anything out — how a
downstream script pins itself to a known dataset version.

In [ ]:
import dvc.api, pandas as pd, io

# find the commit where raw.csv was FIRST tracked -- that is the v1 dataset
hist = !git log --format=%H -- data/raw.csv.dvc
V1 = hist[-1]                      # git log is newest-first, so the last entry is oldest

# read the current version
cur = pd.read_csv(io.StringIO(dvc.api.read("data/raw.csv", repo=".")))

# read the v1 version -- no checkout, no cd, nothing moved on disk
old = pd.read_csv(io.StringIO(dvc.api.read("data/raw.csv", repo=".", rev=V1)))

print("current  :", len(cur), "rows")
print(f"at {V1[:8]}:", len(old), "rows   <- the pre-March dataset")
print("\nand the working folder was never touched:", len(pd.read_csv("data/raw.csv")), "rows on disk")


## Step 4.2 · What changed, and where does it live?

Three inspection commands you will reach for constantly once a project is real.

In [ ]:
!dvc diff HEAD~1 HEAD      # which tracked files changed between two commits
!dvc data status           # modified / not in cache / not in remote, in one view
!dvc status -c             # specifically: am I in sync with the remote?

## Step 4.3 · Experiments without polluting Git history

`dvc exp run` runs the pipeline with overridden params and stores the result as a **hidden
commit**. No branch clutter — you promote only the winner.

In [ ]:
# -S overrides a param for this run only; params.yaml on disk is left alone
!dvc exp run -S n_estimators=40  --name small
!dvc exp run -S n_estimators=250 --name big
!dvc exp show --only-changed        # the table of every attempt, with metrics

> **Reference:** deck slide 41 covers running these at scale — `dvc exp run --queue`,
> `dvc queue start -j 4`, `dvc exp branch` to promote a winner.

## Step 4.4 · Templated stages — stop copy-pasting YAML

`foreach` generates one stage per item. `matrix` generates the cartesian product. Both keep
the "only re-run what went stale" behaviour per generated stage.

In [ ]:
%%writefile src/slice.py
"""Write a per-segment CSV so we have something for `foreach` to iterate over."""
import sys, pandas as pd, os

segment = sys.argv[1]                       # "premium" or "standard"
df = pd.read_csv("data/prepared/train.csv")
sub = df[df.is_premium == (1 if segment == "premium" else 0)]

os.makedirs("data/segments", exist_ok=True)
sub.to_csv(f"data/segments/{segment}.csv", index=False)
print(f"{segment}: {len(sub)} rows")

In [ ]:
# `dvc.yaml` is just YAML, so we add the templated stage under `stages:` properly.
# (Appending text blindly would be fragile -- DVCLive already added a `plots:` section
#  in Part 3, and an appended block would land under the wrong parent key.)
import yaml

cfg = yaml.safe_load(open("dvc.yaml"))
cfg["stages"]["slice"] = {
    "foreach": ["premium", "standard"],          # one generated stage per entry
    "do": {
        "cmd":  "python src/slice.py ${item}",
        "deps": ["src/slice.py", "data/prepared"],
        "outs": ["data/segments/${item}.csv"],
    },
}
yaml.safe_dump(cfg, open("dvc.yaml", "w"), sort_keys=False)

print(open("dvc.yaml").read())

In [ ]:
!dvc repro          # note the generated stage names: slice@premium, slice@standard
!dvc dag            # the graph now branches

`matrix` is the same idea across two axes — six stages from four lines:

```yaml
  train:
    matrix:
      model: [rf, xgb]
      split: [a, b, c]
    cmd: python src/train.py --model ${item.model} --split ${item.split}
    outs: [models/${item.model}-${item.split}.pkl]
```

Values can come from `params.yaml` too (`foreach: ${cities}`), and `dvc repro -j 4` runs
independent stages in parallel.

## Step 4.5 · The run cache — why a stage sometimes doesn't run

DVC records a signature of every stage execution in `.dvc/cache/runs`. If it has seen this
exact combination of command, dependencies and params before, it **restores the outputs**
instead of recomputing them.

Watch it happen: change `max_depth` to a value never used, run it, then change it straight
back to a value we *have* just used.

In [ ]:
# A -> B : max_depth 3 has never been run, so this one really trains
cfg = open("params.yaml").read().replace("max_depth: 8", "max_depth: 3")
open("params.yaml", "w").write(cfg)
!dvc repro train


That one executed. Now go straight back to `max_depth: 8` — the exact configuration DVC
ran moments ago, with the same dependencies.


In [ ]:
# B -> A : this signature is already in .dvc/cache/runs
cfg = open("params.yaml").read().replace("max_depth: 3", "max_depth: 8")
open("params.yaml", "w").write(cfg)

!dvc repro train


Compare the two outputs:

```
Running stage 'train':                                  <- the first one really trained
Stage 'train' is cached - skipping run, checking out outputs   <- the second one did not
```

DVC recognised the signature and took the outputs straight from `.dvc/cache/runs`. That cache
travels with `dvc push` / `dvc pull`, so a colleague who tries a configuration you already ran
gets your results without spending the compute again.

```bash
dvc repro --no-run-cache    # force real execution
dvc repro -f                # force every stage
dvc repro -s train          # a single stage
```

In [ ]:
# make sure the workspace and the lock file agree before moving on
!dvc repro -q
!git add -A && git commit -q -m "chore: params back to max_depth 8"


## Step 4.6 · `.dvcignore`

Same syntax as `.gitignore`, but it tells **DVC** what to skip when it scans a tracked
directory. Useful when a data folder is full of scratch files you do not want hashed.

In [ ]:
%%writefile .dvcignore
# scratch files inside tracked directories -- never hash these
*.tmp
.ipynb_checkpoints/
**/__pycache__/

In [ ]:
!git add .dvcignore && git commit -q -m "chore: add .dvcignore"
!dvc status          # unchanged: the ignore file only affects what DVC scans

## Step 4.7 · `artifacts:` — DVC's own model registry

Add an `artifacts:` block to `dvc.yaml` and your model gets a **name**, a type and labels.
Consumers then fetch it by name and git revision instead of guessing at paths.

In [ ]:
import yaml

cfg = yaml.safe_load(open("dvc.yaml"))
cfg["artifacts"] = {
    "churn-classifier": {                        # the artifact ID
        "path": "models/model.pkl",              # the only required field
        "type": "model",                         # "model" is what the registry lists
        "desc": "Churn random forest, sklearn",
        "labels": ["tabular", "sklearn"],
        "meta": {"framework": "scikit-learn"},
    }
}
yaml.safe_dump(cfg, open("dvc.yaml", "w"), sort_keys=False)

print(yaml.safe_load(open("dvc.yaml"))["artifacts"])

In [ ]:
!git add dvc.yaml && git commit -q -m "feat: declare model artifact"

# DVC reads artifact VERSIONS from git tags shaped  <artifact-name>@<version>
!git tag churn-classifier@v1.0.0
!git tag -l


---
# Part 5 — The real thing: `data.zip`   ·   deck slides 3–6

Everything so far used a small generated CSV so the notebook runs anywhere. Now do it once on
the **actual dataset from the deck** — `data.zip`, 39 MB of cat and dog photographs, which
ships in this repository next to the notebook.

This is the exact scenario slides 3–6 describe: a big zip sitting loose in a folder, no
version, no history. Watch it become a tracked dataset that Git carries in five lines of text.

In [ ]:
import pathlib, zipfile

# data.zip ships with this repo, right next to the notebook. The other paths are fallbacks
# in case you moved it, or are running this notebook outside the repo.
CANDIDATES = [BASE / "data.zip", BASE.parent / "data.zip", BASE.parent.parent / "data.zip",
              pathlib.Path.home() / "data.zip"]
DATA_ZIP = next((p for p in CANDIDATES if p.is_file()), None)

if DATA_ZIP:
    z = zipfile.ZipFile(DATA_ZIP)
    files = [n for n in z.namelist() if not n.endswith("/")]
    print(f"found {DATA_ZIP}")
    print(f"  {DATA_ZIP.stat().st_size/1024/1024:.1f} MB on disk, {len(files)} files inside")
    print("  e.g.", files[1])
else:
    print("data.zip not found — Part 5 will skip itself. Looked in:")
    for p in CANDIDATES: print("   ", p)

## Step 5.1 · Unpack it and track it

`dvc add` on a **folder** behaves exactly like `dvc add` on a file: one pointer for the whole
tree, with a count of how many files are inside.

We unpack into `photos/` rather than `data/`, because this sandbox already uses `data/` for the
pipeline's outputs — and DVC refuses to track a folder that overlaps something it already
tracks. (That error message is a useful one to have seen once.)

In [ ]:
if DATA_ZIP:
    import os, shutil
    # The zip contains a top-level `data/`, but this sandbox already uses `data/` for the
    # pipeline's own tracked outputs -- DVC refuses to track a folder that overlaps them.
    # So unpack the photos into their own directory.
    shutil.rmtree("_unzip", ignore_errors=True); shutil.rmtree("photos", ignore_errors=True)
    z.extractall("_unzip")
    os.rename("_unzip/data", "photos")
    shutil.rmtree("_unzip", ignore_errors=True)

    !du -sh photos/train photos/validation
    print("jpg files:", len(list(pathlib.Path("photos").rglob("*.jpg"))))

In [ ]:
if DATA_ZIP:
    !dvc add photos        # note: a directory, not a single file

## Step 5.2 · What Git is now carrying

The whole photo library is represented by one small text file. Compare the two sizes — this is
the slide-5 claim, reproduced on your own machine.

In [ ]:
if DATA_ZIP:
    print("--- the pointer Git will store ---")
    !cat photos.dvc
    print("--- sizes ---")
    !du -sh photos                     # the photographs
    !du -sh --apparent-size .git       # everything Git is tracking
    !git add photos.dvc .gitignore && git commit -q -m "data: add cats & dogs photo set"
    !git log --oneline -1

## Step 5.3 · Push it, delete it, get it back

The same three commands from Part 1 — now on 39 MB of images instead of a toy CSV.

In [ ]:
if DATA_ZIP:
    !dvc push
    print("\n--- the remote now holds the photographs ---")
    !du -sh "{REMOTE}"

In [ ]:
if DATA_ZIP:
    shutil.rmtree("photos")
    print("deleted photos/ entirely. jpg files on disk:",
          len(list(pathlib.Path(".").rglob("photos/**/*.jpg"))))

    !dvc pull -q
    print("after dvc pull      :", len(list(pathlib.Path("photos").rglob("*.jpg"))),
          "jpg files restored")

### What just happened

* `photos/` is **not** in Git — check `.gitignore`. Git stores `photos.dvc`, five lines of
  text, and the 43 MB of images live in DVC storage.
* A teammate clones the repo and runs `dvc pull` to get the photographs.
* The dataset now has a version. Add more photos, `dvc add photos` again, commit, and you can
  return to today's exact set forever with `git checkout` + `dvc checkout`.

### One honest exception

`data.zip` itself **is** committed to this training repo, so that the tutorial works the moment
you clone it. That is a deliberate exception for teaching, not a recommendation — it is exactly
the thing the last 50 slides argue against, and it costs every clone 39 MB forever.

In a real project you would `dvc add data.zip` (or skip the zip and track `photos/` directly),
push the bytes to a DVC remote, and let Git carry the pointer. That is what you just did in
Steps 5.1–5.3.

> And if a big file has already been committed by mistake, deck slide 44 shows how to remove
> it from history.

> That tag is the release. From another repo a consumer fetches it by name and version:
> `dvc artifacts get <repo-url> churn-classifier --rev v1.0.0` — deck slide 49.

---
# Reference — on the slides, not in this notebook

These need a team, a cloud bucket or a second repo, so they are explained on the deck rather
than run here. Each line names the slide that covers it.

| Topic | Deck slide | One-line version |
|---|---|---|
| Data registries | 39 | `dvc import` / `dvc get` / `dvc update` — pin another repo's dataset by revision |
| Templated & matrix stages | 40 | `foreach:` and `matrix:` generate many stages from one block (you ran `foreach` in 4.4) |
| Experiments at scale | 41 | `dvc exp run --queue` + `dvc queue start -j 4`, then `dvc exp branch` to promote |
| Shared team cache | 42 | `dvc cache dir /mnt/shared` + `cache.shared group` — one copy per server |
| Plots for reviewers | 43 | a `plots:` block in `dvc.yaml`, then `dvc plots diff main workspace` |
| Migrating an existing mess | 44 | `git filter-repo` to purge a committed big file, then `dvc add` it properly |
| **Credentials** | 45 | `dvc remote modify --local` → `.dvc/config.local`, which is git-ignored. **Never put a key in `.dvc/config`** |
| Merge conflicts on data | 47 | `.gitattributes` + the `merge=dvc` driver; resolves append-only dirs, refuses real conflicts |
| The rest of `dvc.yaml` | 48 | `vars`, `wdir`, `frozen`, `always_changed`, `persist`, `push`, nested params |
| DVC 2.x → 3.x | 50 | cache moved to `.dvc/cache/files/md5/…`; `dvc cache migrate`; 2.x cannot read 3.x lock files |

---
# Recap

| You wanted to… | Command |
|---|---|
| put a big file under version control | `dvc add data/raw.csv` then `git add data/raw.csv.dvc` |
| see what changed | `dvc status` (`-c` compares against the remote) · `dvc data status` |
| share the data | `dvc remote add -d storage <url>` · `dvc push` / `dvc pull` |
| return to an old version | `git checkout <sha>` **then** `dvc checkout` |
| read an old version from Python | `dvc.api.read(path, repo=".", rev="HEAD~1")` |
| record how outputs are produced | `dvc.yaml` + `dvc repro` |
| see the stage graph | `dvc dag` |
| generate many similar stages | `foreach:` / `matrix:` in `dvc.yaml` |
| check if the model improved | `dvc metrics show` / `dvc metrics diff <rev>` |
| try ideas without committing | `dvc exp run -S k=v` · `dvc exp show` · `dvc queue start` |
| reuse a dataset across repos | `dvc import` / `dvc get` / `dvc update` |
| free disk space | `dvc gc -w` (careful) |
| log metrics from Python | `dvclive`: `Live()`, `log_metric`, `next_step` |
| skip work already done | the run cache — `dvc repro --no-run-cache` to force |
| store credentials safely | `dvc remote modify --local` (→ `.dvc/config.local`) |
| auto-resolve data merges | `.gitattributes` + `merge=dvc` driver |
| name a model for consumers | `artifacts:` in `dvc.yaml` + `dvc artifacts get` |
| untrack / rename / pin | `dvc remove` · `dvc move` · `dvc freeze` |

## The two rules that prevent every beginner bug

1. **Data changed?** `dvc add` **then** `git add <file>.dvc` — always both, in that order.
2. **Moved in history?** `git checkout` **then** `dvc checkout` — always both, in that order.

## What to do at work tomorrow

1. `dvc init` + `dvc add` on **one** dataset, pushed to a shared folder or bucket. That alone
   kills the `data_final_v3/` folder.
2. Move your steps into `dvc.yaml` the first time you re-run something by hand.
3. Add `dvc metrics diff` to CI so reviewers see the score change on every pull request —
   see `dvc/cml-example.yaml` in this repo for a working one.
4. Only then: `dvc exp`, plots, shared caches.

## Docs

* DVC — <https://dvc.org/doc>
* Command reference — <https://dvc.org/doc/command-reference>
* CML, DVC in CI — <https://cml.dev>

## Cleanup (optional)

Uncomment and run to delete the sandbox and the fake remote. Leaving it is fine too —
`dvc_demo/` is a real git+DVC repo you can keep poking at.

In [ ]:
# import shutil, os
# os.chdir(BASE)
# shutil.rmtree(PROJ, ignore_errors=True)
# shutil.rmtree(REMOTE, ignore_errors=True)
# print("sandbox removed")